# 02 · Features

Three escalating representations of the same draft, built by `src/features.py`:

| Tier | Idea | Shape |
|---|---|---|
| 1 | multi-hot champion indicators per side, plus bans | wide and sparse |
| 2 | archetype aggregates (engage, tank, hard CC, AD/AP …) and blue-minus-red diffs | narrow and dense |
| 3 | *(optional)* champion win-rate deltas learned from outcomes | tiny, and leakage-prone |

Every feature is known the moment champion select locks, so none of them can leak the result.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, parse, features as F, viz
viz.apply_style()
pd.set_option("display.width", 140)

In [2]:
df = F.load_matches()
labels = F.load_labels()
y = df["win"]
print(f"{len(df):,} matches, blue win rate {y.mean():.2%}")

8,690 matches, blue win rate 48.23%


## Tier 1 — multi-hot champions

One indicator column per (champion, side), plus one per banned champion. This throws away
*nothing* about who was picked, which is exactly why it ends up being the best representation
we build — champion identity is where what little signal exists actually lives.

The cost is dimensionality: hundreds of columns, each mostly zero.

In [3]:
X1, y = F.tier1_features(df)
density = X1.to_numpy().mean()
print(f"Tier-1 matrix: {X1.shape[0]:,} rows x {X1.shape[1]} columns")
print(f"non-zero cells: {density:.2%}  (each row has ~{X1.sum(axis=1).mean():.0f} active features)")
X1.iloc[:3, :6]

Tier-1 matrix: 8,690 rows x 519 columns
non-zero cells: 3.60%  (each row has ~19 active features)


,blue_pick_1,red_pick_1,blue_pick_2,red_pick_2,blue_pick_3,red_pick_3
0,0,0,0,0,0,0
1,1,0,0,0,0,0
2,0,0,0,0,0,0


In [4]:
# Decode one row back to champion names, to prove the encoding is what we think it is.
row = X1.iloc[0]
active = [c for c in X1.columns if row[c] == 1]
def pretty(col):
    cid = int(col.split("_")[-1])
    kind = "ban" if col.startswith("ban") else ("blue" if "blue" in col else "red")
    return f"{labels.loc[cid, 'name']} ({kind})"
print("match", df.iloc[0]["matchId"], "->", ", ".join(pretty(c) for c in active))

match SG2_157945922 -> Irelia (blue), Riven (red), Viktor (red), Zoe (blue), Kai'Sa (blue), Seraphine (red), Yasuo (blue), Taliyah (red), Ivern (blue), Smolder (red), Jax (ban), Evelynn (ban), Katarina (ban), Akali (ban), Xerath (ban), Graves (ban), Rengar (ban), Jayce (ban), Pyke (ban), Mel (ban)


## Tier 2 — archetype aggregates

Instead of *who* was picked, describe *what kind* of team it is: how many engage champions,
how much hard CC, the AD/AP split, how many bodies are frontline. Each is computed per team,
plus a signed blue-minus-red difference.

Two honest caveats:

- The archetype labels in `champion_labels.csv` are a **hand-curated first pass** for the four
  subjective columns (engage, enchanter, poke, hard CC). The objective columns come straight
  from Riot's own class tags.
- Aggregating five champions into counts **destroys champion identity**. Notebook 03 shows this
  costs more than it gains.

In [5]:
X2, _ = F.tier2_features(df, labels)
print(f"Tier-2 matrix: {X2.shape[0]:,} rows x {X2.shape[1]} columns (dense)")
X2.filter(like="diff_").describe().T[["mean", "std", "min", "max"]].round(2)

Tier-2 matrix: 8,690 rows x 39 columns (dense)


,mean,std,min,max
diff_is_tank,-0.01,1.14,-4.0,4.0
diff_is_mage,0.03,1.27,-4.0,4.0
diff_is_marksman,0.00,0.86,-3.0,4.0
diff_is_fighter,-0.00,1.10,-4.0,4.0
diff_is_assassin,0.01,1.33,-5.0,5.0
diff_is_support,-0.00,0.93,-3.0,3.0
diff_engage,0.00,1.01,-3.0,4.0
diff_enchanter,-0.01,0.73,-3.0,2.0
diff_poke,0.02,0.93,-3.0,3.0
diff_hard_cc,-0.00,1.12,-4.0,4.0


The `diff_*` columns are centred near zero with a spread of roughly one champion — matchmaking
produces broadly similar team compositions, which is itself a hint that comp differences are
small in practice.

## Tier 3 — champion win-rate deltas *(optional, and a leakage trap)*

Tier 3 sums each champion's empirical win rate. Unlike tiers 1 and 2, these features are
**derived from outcomes**, so they must be computed on training rows only and applied to held-out
rows. Fitting them on the full dataset and then evaluating on that same data would leak the
answer and produce a flattering, meaningless score.

`tier3_features()` takes a `train_idx` for exactly this reason. It is kept as a scaffold rather
than a result.

In [6]:
X3, _ = F.tier3_features(df)   # whole-frame fit: summary only, NOT for evaluation
print(f"Tier-3 matrix: {X3.shape}")
print("\nLeakage warning: fitted on all rows, so these numbers are optimistic by construction.")
X3.head(3)

Tier-3 matrix: (8690, 3)

Leakage warning: fitted on all rows, so these numbers are optimistic by construction.


,blue_champ_wr_sum,red_champ_wr_sum,diff_champ_wr
0,2.400934,2.488636,-0.087702
1,2.308554,2.503115,-0.194561
2,2.320595,2.582501,-0.261906


## Summary

All three tiers align to the same label vector, so they are directly comparable in notebook 03.
Tier 1 keeps the most information; Tier 2 is smaller and more interpretable but discards identity;
Tier 3 is a scaffold that needs careful fold-wise fitting to be legitimate.